<a href="https://colab.research.google.com/github/xingji1337/week5RAG/blob/TrackBMultimodal/Week5_2_RAG_Multimodal_HOME_REPAIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 — Track B: Multimodal RAG (Home Repair AI)

This notebook extends the RAG pipeline into **multimodal retrieval** for the *Home Repair Assistant* app.

Sources include:
- PDFs (repair guides, drywall, water leaks)
- Images (photos, diagrams embedded in PDFs — optional extraction)
- Text queries from users

Features:
- Text + Image embedding retrieval
- RRF fusion across modalities
- Multimodal reranker (optional)
- Context builder mixing text passages and image captions


## 0) Install dependencies

In [ ]:
# Uncomment in Colab/local
!pip install sentence-transformers rank_bm25 faiss-cpu pypdf pillow transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 6.3 MB/s eta 0:00:00


## 1) Mount Drive & set paths

In [ ]:
import os, pathlib
USE_DRIVE=True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/'
PDF_FILES = [
    os.path.join(DATA_DIR, 'Complete home repair  with 350 projects and 2300 photos.pdf'),
    os.path.join(DATA_DIR, '7 Different Ways to Repair Drywall.pdf'),
    os.path.join(DATA_DIR, 'How to stop Water damage when A Leak.pdf')
]

Mounted at /content/drive


## 2) Load text + extract images from PDFs

In [ ]:
from pypdf import PdfReader
from PIL import Image
import io

def extract_text_images(pdf_path):
    reader = PdfReader(pdf_path)
    texts, images = [], []
    for i,page in enumerate(reader.pages):
        txt = page.extract_text() or ""
        if txt.strip():
            texts.append((i, txt))
        # image extraction
        if "/XObject" in page["/Resources"]:
            xObj = page["/Resources"]["/XObject"].get_object()
            for obj in xObj:
                if xObj[obj]["/Subtype"] == "/Image":
                    size = (xObj[obj]["/Width"], xObj[obj]["/Height"])
                    data = xObj[obj].get_data()
                    try:
                        img = Image.open(io.BytesIO(data))
                        images.append((i, img))
                    except:
                        pass
    return texts, images

# Example check
txts, imgs = extract_text_images(PDF_FILES[1])
print("Text chunks:", len(txts), "Images:", len(imgs))


Text chunks: 5 Images: 1


## 3) Build multimodal corpus

In [ ]:
corpus_text, corpus_images = [], []

for path in PDF_FILES:
    t, im = extract_text_images(path)
    corpus_text.extend([(path, p, txt) for p, txt in t])
    corpus_images.extend([(path, p, img) for p, img in im])

print("Corpus sizes:", len(corpus_text), "text chunks,", len(corpus_images), "images")


Corpus sizes: 616 text chunks, 1245 images


## 4) Text retrieval (BM25 + dense)

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

bm25 = BM25Okapi([txt.split() for _,_,txt in corpus_text])
dense_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device='cpu')
dense_embs = dense_model.encode([txt for _,_,txt in corpus_text], convert_to_tensor=True)

def text_search(query, topk=10):
    bm_scores = bm25.get_scores(query.split())
    bm_ranked = sorted(list(enumerate(bm_scores)), key=lambda x:x[1], reverse=True)[:topk]
    dense_query = dense_model.encode([query], convert_to_tensor=True)
    cos = util.cos_sim(dense_query, dense_embs)[0]
    vals, idxs = cos.topk(topk)
    dense_ranked = [(i, float(s)) for i,s in zip(idxs.tolist(), vals.tolist())]
    return bm_ranked, dense_ranked

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 5) Image retrieval (CLIP)

In [ ]:
from sentence_transformers import SentenceTransformer
import torch # Import torch to use torch.cat

clip_model = SentenceTransformer("clip-ViT-B-32", device = "cpu")

def build_image_embs(corpus_images, batch_size=8):
    if not corpus_images:
        print("⚠️ No images found in corpus.")
        return None
    embs = []
    pil_images = [img.convert("RGB").resize((224,224)) for _,_,img in corpus_images]
    for i in range(0, len(pil_images), batch_size):
        batch = pil_images[i:i+batch_size]
        embs.append(clip_model.encode(batch, convert_to_tensor=True))
    return torch.cat(embs, dim=0)

image_embs = build_image_embs(corpus_images, batch_size=8)

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

0_CLIPModel/pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

0_CLIPModel/model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/604 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
print(type(image_embs), image_embs.shape)

## 6) Multimodal fusion (RRF)

In [ ]:
from collections import defaultdict

def rrf(results_lists, K=60, topk=10):
    scores = defaultdict(float)
    for rl in results_lists:
        for rank,(idx,score) in enumerate(rl,1):
            scores[idx]+=1/(K+rank)
    return sorted(scores.items(), key=lambda x:x[1], reverse=True)[:topk]


## 7) Context builder (text+image captions)

In [ ]:
def build_context(query, top_text, top_images):
    ctx = []
    for idx,_ in top_text:
        path, p, txt = corpus_text[idx]
        ctx.append(f"[Text {path}#p{p}] {txt[:400]}")
    for idx,_ in top_images:
        path, p, _ = corpus_images[idx]
        ctx.append(f"[Image {path}#p{p}] (image retrieved relevant to query)")
    return "\n\n".join(ctx)


## 8) Full multimodal pipeline

In [ ]:
def run_mm_pipeline(query):
    bm, dense = text_search(query)
    fused_text = rrf([bm,dense], topk=5)
    top_images = image_search(query, topk=3)
    ctx = build_context(query, fused_text, top_images)
    return ctx, fused_text, top_images

q = "How do I repair drywall and what does it look like?"
ctx, ft, fi = run_mm_pipeline(q)
print(ctx)


## 9) Qualitative examples

In [ ]:
qs = [
    "How do I repair a small hole in drywall?",
    "Show me what water damage signs look like",
]
for q in qs:
    ctx,ft,fi=run_mm_pipeline(q)
    print("\nQ:",q,"\n",ctx[:500])


## 10) Save run config

In [ ]:
out_dir="./week5_outputs_mm"
os.makedirs(out_dir,exist_ok=True)
cfg={"text_model":"all-MiniLM-L6-v2","image_model":"clip-ViT-B-32","fusion":"RRF"}
import json
with open(os.path.join(out_dir,"mm_config.json"),"w") as f: json.dump(cfg,f,indent=2)
print("Saved config.")
